In [1]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Setup & Load Environment
load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

# 2. Gemini-adapted Completion Function
def get_completion_from_messages(messages, 
                                 model="gemini-3.1-flash-lite-preview", 
                                 temperature=0, 
                                 max_tokens=500):
    # Extract system instruction for Gemini architecture
    system_instr = next((m['content'] for m in messages if m['role'] == 'system'), "")
    
    # Format history (mapping 'assistant' to 'model')
    history = []
    for m in messages:
        if m['role'] == 'system': continue
        role = "model" if m['role'] == 'assistant' else "user"
        history.append({"role": role, "parts": [m['content']]})

    gemini_model = genai.GenerativeModel(
        model_name=model,
        system_instruction=system_instr,
        generation_config={
            "temperature": temperature,
            "max_output_tokens": max_tokens,
        }
    )

    # Start chat and send the prompt
    chat = gemini_model.start_chat(history=history[:-1])
    response = chat.send_message(history[-1]["parts"][0])
    return response.text.strip()

# 3. Prepare Data for Evaluation (The "Y/N" Logic)
system_message = """
You are an assistant that evaluates whether customer service agent responses \
sufficiently answer questions and validates facts from product information.
Respond with a Y or N character only:
Y - if the output sufficiently answers AND facts are correct
N - otherwise
"""

product_information = """
{ "name": "SmartX ProPhone", "features": ["6.1-inch display", "5G"] }
{ "name": "FotoSnap DSLR Camera", "features": ["24.2MP sensor"] }
""" # Simplified for example

customer_message = "tell me about the smartx pro phone and the fotosnap camera."
final_response_to_customer = "The SmartX ProPhone has a 6.1-inch display and 5G. The FotoSnap DSLR has a 24.2MP sensor."

q_a_pair = f"""
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{final_response_to_customer}```

Output Y or N
"""

# 4. Execute Evaluation
messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user', 'content': q_a_pair}
]

evaluation_result = get_completion_from_messages(messages, max_tokens=1)
print(f"Evaluation Result: {evaluation_result}")

# Example of further process based on Y/N
if evaluation_result == "Y":
    print("Response verified. Sending to customer.")
else:
    print("Response failed validation. Regenerating...")

C:\Anaconda3\envs\lang_chain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_7572\2288500485.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Evaluation Result: 
Response failed validation. Regenerating...
